# 04 · Landmark extraction handoff and validation

**Adapted:** MediaPipe extraction is already complete. Current arrays are float32 `(T,225)` with pose/left hand/right hand coordinates concatenated. This notebook validates the delivered arrays; the previously planned RGB-frame extraction stage is complete for the current dataset.


In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd().resolve()
while not (ROOT/'data/processed/WLASL100/all_metadata.csv').exists() and ROOT.parent!=ROOT: ROOT=ROOT.parent
DATA=ROOT/'data/processed/WLASL100'
if not (DATA/'all_metadata.csv').exists(): raise FileNotFoundError('Open notebook from the sign-language-recognition repository')
import numpy as np
parts=[]
for src in ('landmarks_repo','landmarks_kaggle'):
 idx=pd.read_csv(DATA/src/'index.csv',dtype={'video_id':str}); idx['source']=src
 parts.append(idx)
index=pd.concat(parts,ignore_index=True)
checks=[]
for r in index.itertuples():
 p=DATA/r.source/f'{str(r.video_id).zfill(5)}.npy'; a=np.load(p,mmap_mode='r')
 checks.append({'video_id':str(r.video_id).zfill(5),'source':r.source,'ok':a.ndim==2 and a.shape[1]==225 and a.dtype==np.float32,'frames':a.shape[0],'nan_pct':100*np.isnan(a).mean()})
qc=pd.DataFrame(checks)
assert qc.ok.all(), qc.loc[~qc.ok].head()
qc.to_csv(DATA/'landmarks_eda'/'landmark_array_validation.csv',index=False)
print(f"validated {len(qc)} arrays; shapes (T,225); median NaN rate {qc.nan_pct.median():.2f}%")
